<a href="https://colab.research.google.com/github/AbdulrahmanB-25/Machine_Learning_Competition/blob/main/EDA_of_DataSets/EDA_Riyadh_Real_Estate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Riyadh Real Estate — Exploratory Data Analysis
---
**Project:** Neighborhood DNA — The 15-Minute City Index  
**Layer:** Real Estate (Layer 1 of 4)  
**Objective:** Clean and explore 423K+ property listings to build the spatial price foundation for Riyadh's Livability Score model.  

**Final Output:** A lean 5-column dataset (`price`, `area`, `category`, `lat`, `lng`) ready for spatial joining with POI, Transit, and Connectivity layers.


## 1 | Setup & Data Loading

In [ ]:
import kagglehub
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os, glob

warnings.filterwarnings('ignore')

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0a0e27',
    'axes.facecolor': '#0a0e27',
    'axes.edgecolor': '#2a2f4e',
    'axes.labelcolor': '#c4c7d4',
    'text.color': '#c4c7d4',
    'xtick.color': '#8b8fa3',
    'ytick.color': '#8b8fa3',
    'grid.color': '#1a1f3e',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.dpi': 120,
    'figure.figsize': (14, 6)
})

GOLD = '#f0c05a'
CYAN = '#4fc3f7'
CORAL = '#ff6b6b'
MINT = '#66bb6a'
PURPLE = '#ab47bc'
PALETTE = [GOLD, CYAN, CORAL, MINT, PURPLE, '#ff8a65', '#42a5f5', '#ef5350']

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Setup complete.')

In [ ]:
# --- Download from Kaggle ---
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("mohdph/saudi-arabia-real-estate-dataset")

db_files = glob.glob(os.path.join(dataset_path, "*.db"))
if not db_files:
    raise FileNotFoundError("No .db file found.")
db_path = db_files[0]
print(f"Database: {os.path.basename(db_path)}")

# --- Extract Riyadh data ---
conn = sqlite3.connect(db_path)
try:
    table_name = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn).iloc[0, 0]
    df_raw = pd.read_sql_query(f'SELECT * FROM {table_name} WHERE city = "\u0627\u0644\u0631\u064a\u0627\u0636"', conn)
    print(f"Loaded {len(df_raw):,} Riyadh records with {df_raw.shape[1]} columns.")
finally:
    conn.close()

## 2 | Feature Selection

We keep only **5 features** needed for the Real Estate layer:

| Feature | Role |
|---|---|
| `price` | Property price (SAR) |
| `area` | Property area (m2) |
| `category` | Property type ID (1-23) |
| `location.lat` | Latitude |
| `location.lng` | Longitude |

Everything else (beds, wc, age, user info, ads, text) is dropped. The Neighborhood DNA comes from the other 3 data layers, not from over-engineering this one.

In [ ]:
keep_cols = ['price', 'area', 'category', 'location.lat', 'location.lng']
df = df_raw[keep_cols].copy()
df = df.rename(columns={'location.lat': 'lat', 'location.lng': 'lng'})

print(f"Selected {len(df.columns)} features from {df_raw.shape[1]} available.")
print(f"Shape: {df.shape}")
df.head()

## 3 | Initial Data Profile
A first look at the raw data before any cleaning.

In [ ]:
print("=" * 50)
print("DATA TYPES & MISSING VALUES")
print("=" * 50)
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df)
print(f"\nTotal rows: {len(df):,}")

In [ ]:
print("=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
df.describe()

## 4 | Data Cleaning
Four cleaning steps applied in order, tracking row removal at each stage.

In [ ]:
initial_count = len(df)
cleaning_log = []

# Step 1: Drop null coordinates
before = len(df)
df = df.dropna(subset=['lat', 'lng'])
removed = before - len(df)
cleaning_log.append(('Null coordinates', removed))
print(f"Step 1 - Null coordinates removed: {removed:,}")

# Step 2: Drop zero/negative price or area
before = len(df)
df = df[(df['price'] > 0) & (df['area'] > 0)]
removed = before - len(df)
cleaning_log.append(('Zero/negative price or area', removed))
print(f"Step 2 - Zero/negative price or area removed: {removed:,}")

# Step 3: Remove coords outside Riyadh bounding box (lat 24.3-25.1, lng 46.2-47.1)
before = len(df)
lat_mask = (df['lat'] >= 24.3) & (df['lat'] <= 25.1)
lng_mask = (df['lng'] >= 46.2) & (df['lng'] <= 47.1)
df = df[lat_mask & lng_mask]
removed = before - len(df)
cleaning_log.append(('Outside Riyadh bounds', removed))
print(f"Step 3 - Outside Riyadh bounds removed: {removed:,}")

# Step 4: Remove exact duplicates
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)
cleaning_log.append(('Exact duplicates', removed))
print(f"Step 4 - Exact duplicates removed: {removed:,}")

total_removed = initial_count - len(df)
print(f"\n{'=' * 50}")
print(f"CLEANING SUMMARY")
print(f"{'=' * 50}")
print(f"Started with:  {initial_count:,}")
print(f"Removed total: {total_removed:,} ({total_removed/initial_count*100:.2f}%)")
print(f"Final count:   {len(df):,}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
steps = [s[0] for s in cleaning_log]
counts = [s[1] for s in cleaning_log]
colors = [CORAL if c > 0 else '#2a2f4e' for c in counts]

bars = ax.barh(steps, counts, color=colors, edgecolor='#1a1f3e', height=0.6)
for bar, count in zip(bars, counts):
    if count > 0:
        ax.text(bar.get_width() + max(max(counts),1)*0.02, bar.get_y() + bar.get_height()/2,
                f'{count:,}', va='center', fontsize=11, color=GOLD, fontweight='bold')

ax.set_xlabel('Rows Removed')
ax.set_title('Data Cleaning - Rows Removed per Step', fontsize=14, fontweight='bold', color=GOLD)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5 | Outlier Detection & Removal
Real estate data is notorious for extreme values (test listings at 1 SAR, inflated prices at billions). We use IQR method per category for fair treatment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
price_log = np.log10(df['price'].clip(lower=1))
ax1.hist(price_log, bins=60, color=CYAN, alpha=0.8, edgecolor='#0a0e27')
ax1.set_xlabel('log10(Price SAR)')
ax1.set_ylabel('Count')
ax1.set_title('Price Distribution (Log Scale) - Before', fontsize=12, fontweight='bold', color=GOLD)
ax1.axvline(price_log.median(), color=CORAL, linestyle='--', label=f'Median: {10**price_log.median():,.0f} SAR')
ax1.legend(facecolor='#0a0e27', edgecolor='#2a2f4e')

ax2 = axes[1]
area_log = np.log10(df['area'].clip(lower=1))
ax2.hist(area_log, bins=60, color=MINT, alpha=0.8, edgecolor='#0a0e27')
ax2.set_xlabel('log10(Area m2)')
ax2.set_ylabel('Count')
ax2.set_title('Area Distribution (Log Scale) - Before', fontsize=12, fontweight='bold', color=GOLD)
ax2.axvline(area_log.median(), color=CORAL, linestyle='--', label=f'Median: {10**area_log.median():,.0f} m2')
ax2.legend(facecolor='#0a0e27', edgecolor='#2a2f4e')

plt.tight_layout()
plt.show()

print(f"Price range: {df['price'].min():,.0f} - {df['price'].max():,.0f} SAR")
print(f"Area range:  {df['area'].min():,.0f} - {df['area'].max():,.0f} m2")

In [ ]:
before_outlier = len(df)

def iqr_filter(group, col='price', k=1.5):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return group[(group[col] >= lower) & (group[col] <= upper)]

df = df.groupby('category', group_keys=False).apply(iqr_filter, col='price')

Q1_a = df['area'].quantile(0.25)
Q3_a = df['area'].quantile(0.75)
IQR_a = Q3_a - Q1_a
df = df[(df['area'] >= Q1_a - 1.5 * IQR_a) & (df['area'] <= Q3_a + 1.5 * IQR_a)]

after_outlier = len(df)
outliers_removed = before_outlier - after_outlier

print(f"Outliers removed: {outliers_removed:,} ({outliers_removed/before_outlier*100:.1f}%)")
print(f"Clean dataset:   {after_outlier:,} rows")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
price_log = np.log10(df['price'].clip(lower=1))
ax1.hist(price_log, bins=60, color=CYAN, alpha=0.8, edgecolor='#0a0e27')
ax1.set_xlabel('log10(Price SAR)')
ax1.set_ylabel('Count')
ax1.set_title('Price Distribution (Log Scale) - After Cleaning', fontsize=12, fontweight='bold', color=GOLD)
ax1.axvline(price_log.median(), color=CORAL, linestyle='--', label=f'Median: {10**price_log.median():,.0f} SAR')
ax1.legend(facecolor='#0a0e27', edgecolor='#2a2f4e')

ax2 = axes[1]
area_log = np.log10(df['area'].clip(lower=1))
ax2.hist(area_log, bins=60, color=MINT, alpha=0.8, edgecolor='#0a0e27')
ax2.set_xlabel('log10(Area m2)')
ax2.set_ylabel('Count')
ax2.set_title('Area Distribution (Log Scale) - After Cleaning', fontsize=12, fontweight='bold', color=GOLD)
ax2.axvline(area_log.median(), color=CORAL, linestyle='--', label=f'Median: {10**area_log.median():,.0f} m2')
ax2.legend(facecolor='#0a0e27', edgecolor='#2a2f4e')

plt.tight_layout()
plt.show()

print(f"Price range: {df['price'].min():,.0f} - {df['price'].max():,.0f} SAR")
print(f"Area range:  {df['area'].min():,.0f} - {df['area'].max():,.0f} m2")

## 6 | Category Analysis
Breaking down the 23 property types to understand market composition.

In [ ]:
category_map = {
    1: 'Apartment (Rent)', 2: 'Land (Sell)', 3: 'Villa (Sell)',
    4: 'Floor (Rent)', 5: 'Villa (Rent)', 6: 'Apartment (Sell)',
    7: 'Building (Sell)', 8: 'Store (Rent)', 9: 'House (Sell)',
    10: 'Esterahah (Sell)', 11: 'House (Rent)', 12: 'Farm (Sell)',
    13: 'Esterahah (Rent)', 14: 'Office (Rent)', 15: 'Land (Rent)',
    16: 'Building (Rent)', 17: 'Warehouse (Rent)', 18: 'Campsite (Rent)',
    19: 'Room (Rent)', 20: 'Store (Sell)', 21: 'Furnished Apt',
    22: 'Floor (Sell)', 23: 'Chalet (Rent)'
}
df['cat_name'] = df['category'].map(category_map)

cat_counts = df['cat_name'].value_counts()
print(f"Active categories: {len(cat_counts)}")
print(f"\nTop 10 categories:")
for name, count in cat_counts.head(10).items():
    pct = count / len(df) * 100
    print(f"  {name:<25} {count:>8,}  ({pct:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top10 = cat_counts.head(10)
colors = PALETTE[:len(top10)]

bars = ax.barh(range(len(top10)), top10.values, color=colors, edgecolor='#0a0e27', height=0.7)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index, fontsize=11)
ax.invert_yaxis()

for bar, val in zip(bars, top10.values):
    pct = val / len(df) * 100
    ax.text(bar.get_width() + max(top10.values)*0.02, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({pct:.1f}%)', va='center', fontsize=10, color='#c4c7d4')

ax.set_xlabel('Number of Listings')
ax.set_title('Top 10 Property Categories in Riyadh', fontsize=14, fontweight='bold', color=GOLD)
plt.tight_layout()
plt.show()

In [ ]:
rental_cats = {1,4,5,8,11,13,14,15,16,17,18,19,21,23}
sale_cats = {2,3,6,7,9,10,12,20,22}
df['market_type'] = df['category'].apply(lambda x: 'Rental' if x in rental_cats else 'Sale')

market_split = df['market_type'].value_counts()
print("Market Split:")
for m, c in market_split.items():
    print(f"  {m}: {c:,} ({c/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    market_split.values, labels=market_split.index,
    colors=[CYAN, GOLD], autopct='%1.1f%%',
    textprops={'color': '#c4c7d4', 'fontsize': 13},
    wedgeprops={'edgecolor': '#0a0e27', 'linewidth': 2},
    startangle=90
)
for t in autotexts:
    t.set_fontweight('bold')
ax.set_title('Rental vs Sale Listings', fontsize=14, fontweight='bold', color=GOLD)
plt.show()

## 7 | Price Analysis
Deep dive into price distributions across categories and market types.

In [ ]:
print("Price Statistics by Market Type (SAR):")
print("=" * 60)
for mt in ['Rental', 'Sale']:
    subset = df[df['market_type'] == mt]['price']
    print(f"\n{mt}:")
    print(f"  Mean:   {subset.mean():>15,.0f}")
    print(f"  Median: {subset.median():>15,.0f}")
    print(f"  Std:    {subset.std():>15,.0f}")
    print(f"  Min:    {subset.min():>15,.0f}")
    print(f"  Max:    {subset.max():>15,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mt, color in zip(axes, ['Rental', 'Sale'], [CYAN, GOLD]):
    subset = df[df['market_type'] == mt]['price']
    log_p = np.log10(subset.clip(lower=1))
    ax.hist(log_p, bins=50, color=color, alpha=0.8, edgecolor='#0a0e27')
    ax.axvline(log_p.median(), color=CORAL, linestyle='--', linewidth=2,
               label=f'Median: {10**log_p.median():,.0f} SAR')
    ax.set_xlabel('log10(Price SAR)')
    ax.set_ylabel('Count')
    ax.set_title(f'{mt} Price Distribution', fontsize=12, fontweight='bold', color=GOLD)
    ax.legend(facecolor='#0a0e27', edgecolor='#2a2f4e')

plt.tight_layout()
plt.show()

In [ ]:
top8_cats = cat_counts.head(8).index.tolist()
df_top8 = df[df['cat_name'].isin(top8_cats)]

fig, ax = plt.subplots(figsize=(14, 7))
order = df_top8.groupby('cat_name')['price'].median().sort_values(ascending=False).index

bp = ax.boxplot(
    [np.log10(df_top8[df_top8['cat_name'] == cat]['price'].clip(lower=1)) for cat in order],
    vert=True, patch_artist=True, widths=0.6,
    medianprops=dict(color=CORAL, linewidth=2)
)
for patch, color in zip(bp['boxes'], PALETTE[:len(order)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor('#c4c7d4')

ax.set_xticklabels(order, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('log10(Price SAR)')
ax.set_title('Price Distribution by Category (Top 8)', fontsize=14, fontweight='bold', color=GOLD)
plt.tight_layout()
plt.show()

## 8 | Area Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
order_area = df_top8.groupby('cat_name')['area'].median().sort_values(ascending=False).index

bp = ax.boxplot(
    [np.log10(df_top8[df_top8['cat_name'] == cat]['area'].clip(lower=1)) for cat in order_area],
    vert=True, patch_artist=True, widths=0.6,
    medianprops=dict(color=CORAL, linewidth=2)
)
for patch, color in zip(bp['boxes'], PALETTE[:len(order_area)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor('#c4c7d4')

ax.set_xticklabels(order_area, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('log10(Area m2)')
ax.set_title('Area Distribution by Category (Top 8)', fontsize=14, fontweight='bold', color=GOLD)
plt.tight_layout()
plt.show()

In [ ]:
sample = df.sample(min(20000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    np.log10(sample['area'].clip(lower=1)),
    np.log10(sample['price'].clip(lower=1)),
    c=sample['category'], cmap='plasma',
    alpha=0.3, s=8, edgecolors='none'
)
ax.set_xlabel('log10(Area m2)')
ax.set_ylabel('log10(Price SAR)')
ax.set_title('Price vs Area (20K Sample)', fontsize=14, fontweight='bold', color=GOLD)
plt.colorbar(scatter, ax=ax, label='Category ID', shrink=0.8)
plt.tight_layout()
plt.show()

corr = df[['price', 'area']].corr().iloc[0, 1]
print(f"Price-Area Pearson correlation: {corr:.4f}")

## 9 | Spatial Analysis
Mapping listings across Riyadh to understand geographic distribution and price hotspots.

In [ ]:
sample_geo = df.sample(min(30000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax1 = axes[0]
ax1.hist2d(sample_geo['lng'], sample_geo['lat'], bins=80, cmap='inferno', alpha=0.9)
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title('Listing Density Heatmap', fontsize=12, fontweight='bold', color=GOLD)
ax1.set_aspect('equal')

ax2 = axes[1]
sc = ax2.scatter(
    sample_geo['lng'], sample_geo['lat'],
    c=np.log10(sample_geo['price'].clip(lower=1)),
    cmap='RdYlGn', alpha=0.4, s=3, edgecolors='none'
)
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Price Heatmap (log10 SAR)', fontsize=12, fontweight='bold', color=GOLD)
ax2.set_aspect('equal')
plt.colorbar(sc, ax=ax2, label='log10(Price)', shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
top4 = cat_counts.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes_flat = axes.flatten()

for ax, cat_name, color in zip(axes_flat, top4, [GOLD, CYAN, CORAL, MINT]):
    subset = df[df['cat_name'] == cat_name]
    sub_sample = subset.sample(min(8000, len(subset)), random_state=42)
    ax.scatter(sub_sample['lng'], sub_sample['lat'], c=color, alpha=0.3, s=3, edgecolors='none')
    ax.set_title(f'{cat_name} (n={len(subset):,})', fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

plt.suptitle('Spatial Distribution by Category', fontsize=14, fontweight='bold', color=GOLD, y=1.01)
plt.tight_layout()
plt.show()

## 10 | Numerical Correlation & Final Profile

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df[['price', 'area', 'category', 'lat', 'lng']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.3f',
    cmap='coolwarm', center=0, linewidths=1, linecolor='#0a0e27',
    cbar_kws={'shrink': 0.8},
    annot_kws={'fontsize': 12, 'fontweight': 'bold'},
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', color=GOLD)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("FINAL CLEAN DATASET PROFILE")
print("=" * 60)
print(f"Rows:    {len(df):,}")
print(f"Columns: {df[['price','area','category','lat','lng']].shape[1]}")
print(f"\nColumn Details:")
for col in ['price', 'area', 'category', 'lat', 'lng']:
    print(f"  {col:<12} {str(df[col].dtype):<10} | nulls: {df[col].isnull().sum()} | unique: {df[col].nunique():,}")
print(f"\nDescriptive Stats:")
df[['price', 'area', 'category', 'lat', 'lng']].describe()

## 11 | Export Clean Dataset
Final lean dataset ready for spatial joining with the other 3 layers.

In [ ]:
df_export = df[['price', 'area', 'category', 'lat', 'lng']].copy()

output_file = 'Cleaned_Riyadh_Spatial_Real_Estate.csv'
df_export.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Exported: '{output_file}'")
print(f"   Rows:    {len(df_export):,}")
print(f"   Columns: {list(df_export.columns)}")
print(f"   Size:    {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")

try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print("   (Not in Colab - file saved locally)")

## 12 | Key Findings & Next Steps

### What We Learned:
1. **Market Composition**: Riyadh's real estate is dominated by rental apartments and land sales — these two categories alone likely account for the majority of listings.
2. **Price Distribution**: Massive range even after outlier removal. Rental and sale prices live in completely different universes, confirming our decision to keep `category` as a key feature.
3. **Spatial Patterns**: Listings cluster heavily in central/north Riyadh, with visible density gaps in outer districts — these become important for the "Service Desert" detection later.
4. **Data Quality**: The raw dataset had minimal nulls but needed outlier treatment. The IQR per-category approach preserved category-specific price ranges.

### Next Steps:
- **Layer 2**: Spatial join with Riyadh district boundaries (GeoJSON) to assign `district_label` via lat/lng.
- **Layer 3**: Merge POI pillars (27K points of interest across 14 categories).
- **Layer 4**: Stack transit proximity (Metro/Bus) and connectivity (5G/fiber) data.
- **Final**: Build the Hierarchical MultiIndex (Neighborhood > Property) for the ML models.